# StyleTTS 2: inferencia single-speaker y multi-speaker en macOS

La arquitectura se detecta automáticamente en el YAML del entrenamiento. Selecciona el YAML y checkpoint de la misma ejecución; no cambies `multispeaker` para convertir pesos.

- **Multi-speaker:** utiliza una referencia por voz.
- **Single-speaker con referencia:** genera estilo sin condicionar el difusor con el WAV, y lo mezcla con el estilo extraído del audio mediante alpha/beta.
- **Single-speaker sin referencia:** genera el estilo a partir del texto y ruido; no carga ningún WAV. Alpha/beta no intervienen.

En la configuración, `SINGLE_USE_REFERENCE` controla la prueba single-speaker. Para cambiar de checkpoint reinicia el kernel y ejecuta hasta **Prueba de inferencia**. Las demostraciones originales posteriores son opcionales y usan referencias explícitas.

Instalación y archivos necesarios: [guía de macOS](INFERENCE_MULTISPEAKER_MACOS.md). Se requiere Stage 2 con predictor y difusión entrenados.


### Utils

In [1]:
import torch
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import random
random.seed(0)

import numpy as np
np.random.seed(0)

In [2]:
from pathlib import Path
import os
# Funciona arrancando Jupyter desde la raíz del repo o desde Demo.
ROOT = Path.cwd().resolve()
if not (ROOT / "models.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "models.py").is_file():
    raise RuntimeError("Abre Jupyter desde la raíz de StyleTTS2 o desde Demo")
os.chdir(ROOT)
print(ROOT)

/Users/jairo/Documents/styletts2-valenciano-main/StyleTTS2


In [3]:
# load packages
import time
import random
import yaml
from munch import Munch
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torchaudio
import librosa
from nltk.tokenize import word_tokenize as _word_tokenize
# preserve_line evita descargar punkt/punkt_tab y conserva Treebank del original.
def word_tokenize(text):
    return _word_tokenize(text, preserve_line=True)

from models import build_model, load_ASR_models, load_F0_models
# Evitamos utils.py: monotonic_align sólo es necesario para entrenamiento.
from munch import munchify as recursive_munch
import soundfile as sf
import IPython.display as ipd
from IPython.display import display
from text_utils import TextCleaner
textclenaer = TextCleaner()

%matplotlib inline

/Users/jairo/miniconda3/envs/styletts2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


177


In [4]:
mean, std = -4, 4

def length_to_mask(lengths):
    return torch.arange(int(lengths.max().item()), device=lengths.device)[None, :] >= lengths[:, None]

def preprocess(wave):
    mel_tensor = to_mel(torch.from_numpy(wave).float())
    return (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std

def compute_style(path):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Falta el audio de referencia: {path}")
    wave, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    audio, _ = librosa.effects.trim(wave, top_db=30)
    if len(audio) < SAMPLE_RATE // 2 or not np.isfinite(audio).all() or np.max(np.abs(audio)) < 1e-6:
        raise ValueError("Referencia vacía, demasiado corta o sin voz. Usa varios segundos de voz limpia.")
    mel_tensor = preprocess(audio).to(device)
    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
        ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))
    return torch.cat([ref_s, ref_p], dim=1)

In [5]:
from pathlib import Path

print(Path.cwd())

/Users/jairo/Documents/styletts2-valenciano-main/StyleTTS2


In [44]:
# Edita estas rutas: relativas a ROOT, o absolutas de tu Mac.
CONFIG_PATH = Path("config_valenciano_politica_multi.yml")
CHECKPOINT_PATH = Path("epoch_2nd_00100_gplsi_2.pth")
# Sólo afecta a single-speaker; multi-speaker siempre usa referencia.
SINGLE_USE_REFERENCE = True  # False para generar sin WAV
PLBERT_OVERRIDE = None  # directorio local o ID HF del PL-BERT usado al entrenar
LANGUAGE = "ca-va"  # usa exactamente el idioma/frontend del entrenamiento; LibriTTS: en-us
# CPU es la opción conservadora para Mac Intel y Apple Silicon.
# MPS es opcional: puede fallar en operaciones del vocoder (especialmente iSTFT).
DEVICE = "cpu"  # "mps" para probar Apple Silicon, "cuda" para una máquina NVIDIA
device = torch.device(DEVICE)
if DEVICE == "mps" and not torch.backends.mps.is_available():
    raise RuntimeError("MPS no está disponible; usa DEVICE='cpu'")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible; usa DEVICE='cpu'")
print("Dispositivo:", device)

Dispositivo: cpu


### Load models

In [45]:
import os
import sys
from pathlib import Path
import phonemizer
from phonemizer.backend.espeak.wrapper import EspeakWrapper

# Jupyter/VS Code puede no heredar las variables de la terminal.
# Respeta una ruta explícita; si falta, busca Homebrew en Apple Silicon e Intel.
library = os.environ.get("PHONEMIZER_ESPEAK_LIBRARY")
if not library and sys.platform == "darwin":
    library = next((str(p) for p in (
        Path("/opt/homebrew/opt/espeak-ng/lib/libespeak-ng.dylib"),
        Path("/usr/local/opt/espeak-ng/lib/libespeak-ng.dylib"),
    ) if p.is_file()), None)
if library:
    if not Path(library).is_file():
        raise FileNotFoundError(f"Biblioteca eSpeak inexistente: {library}")
    EspeakWrapper.set_library(library)
    print("eSpeak:", library)
if not phonemizer.backend.EspeakBackend.is_available():
    raise RuntimeError("No se encuentra eSpeak. Instala brew install espeak-ng y configura PHONEMIZER_ESPEAK_LIBRARY con la ruta a su biblioteca.")
global_phonemizer = phonemizer.backend.EspeakBackend(
    language=LANGUAGE, preserve_punctuation=True, with_stress=True)
print("Idioma:", LANGUAGE)


eSpeak: /opt/homebrew/opt/espeak-ng/lib/libespeak-ng.dylib
Idioma: ca-va


In [46]:
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)
if not isinstance(config['model_params'].get('multispeaker'), bool):
    raise ValueError("El YAML debe declarar model_params.multispeaker: true o false")
print("Arquitectura:", "multi-speaker" if config['model_params']['multispeaker'] else "single-speaker")
# Corrige aquí rutas Linux del entrenamiento si las hay. No se usa data_params.root_path.
ASR_CONFIG = Path("Utils/ASR/config.yml")
ASR_WEIGHTS = Path("Utils/ASR/epoch_00080.pth")
F0_WEIGHTS = Path("Utils/JDC/bst.t7")
BERT_PATH = PLBERT_OVERRIDE or config.get('PLBERT_dir', 'Utils/PLBERT/')
for path in (CHECKPOINT_PATH, ASR_CONFIG, ASR_WEIGHTS, F0_WEIGHTS):
    if not path.is_file():
        raise FileNotFoundError(path)
if Path(BERT_PATH).is_dir() and not list(Path(BERT_PATH).glob('step_*.t7')):
    raise FileNotFoundError(f"Faltan pesos step_*.t7 en {BERT_PATH}; revisa PLBERT_OVERRIDE")
text_aligner = load_ASR_models(str(ASR_WEIGHTS), str(ASR_CONFIG))
pitch_extractor = load_F0_models(str(F0_WEIGHTS))
from Utils.PLBERT.util import load_plbert
plbert = load_plbert(str(BERT_PATH))
SAMPLE_RATE = int(config['preprocess_params']['sr'])
spec = config['preprocess_params']['spect_params']
# Conserva los defaults del mel original; dimensiones tomadas del entrenamiento.
to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=config['model_params']['n_mels'], n_fft=spec['n_fft'],
    win_length=spec['win_length'], hop_length=spec['hop_length'])

Arquitectura: multi-speaker
Cargando PL-BERT desde: Utils/PLBERT/


In [47]:
model_params = recursive_munch(config['model_params'])
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]

/Users/jairo/miniconda3/envs/styletts2/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
/Users/jairo/miniconda3/envs/styletts2/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [48]:
# Sólo checkpoints propios o de confianza (torch.load deserializa pickle).
import zipfile
# Los checkpoints ZIP de PyTorch necesitan el directorio del final del archivo.
# Detecta copias incompletas antes de intentar deserializar los pesos.
with CHECKPOINT_PATH.open("rb") as checkpoint_file:
    header = checkpoint_file.read(4)
if header == b"PK\x03\x04":
    try:
        with zipfile.ZipFile(CHECKPOINT_PATH) as archive:
            archive.infolist()
    except zipfile.BadZipFile as exc:
        raise RuntimeError(
            f"Checkpoint incompleto o dañado: {CHECKPOINT_PATH.resolve()} "
            f"({CHECKPOINT_PATH.stat().st_size:,} bytes). "
            "Vuelve a copiar un checkpoint cuyo guardado haya terminado y compara "
            "su SHA-256 con el del equipo de entrenamiento."
        ) from exc
params_whole = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
params = params_whole['net']


In [49]:
# Fallar ante módulos ausentes/incompatibles evita sintetizar con pesos aleatorios.
required = ('bert', 'bert_encoder', 'predictor', 'decoder', 'text_encoder',
            'predictor_encoder', 'style_encoder', 'diffusion')
for key in required:
    if key not in params:
        raise KeyError(f"Falta {key}: comprueba que sea un checkpoint Stage 2 completo")
    state = {k.removeprefix('module.'): v for k, v in params[key].items()}
    model[key].load_state_dict(state, strict=True)
    model[key].eval()
    print(key, 'loaded')

bert loaded
bert_encoder loaded
predictor loaded
decoder loaded
text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded


In [50]:
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

In [51]:
sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0), # empirical parameters
    clamp=False
)

### Synthesize speech

In [52]:
def sample_style(embedding, ref_s=None, diffusion_steps=5, embedding_scale=1):
    """La referencia condiciona el difusor únicamente en multi-speaker."""
    if model_params.multispeaker and ref_s is None:
        raise ValueError("El modelo multi-speaker necesita un audio de referencia")
    if ref_s is not None and tuple(ref_s.shape) != (1, 2 * model_params.style_dim):
        raise ValueError("Dimensiones de estilo de referencia incompatibles con el modelo")
    kwargs = dict(
        noise=torch.randn(1, 1, 2 * model_params.style_dim, device=device),
        embedding=embedding, embedding_scale=embedding_scale, num_steps=diffusion_steps,
    )
    if model_params.multispeaker:
        kwargs['features'] = ref_s
    return sampler(**kwargs).squeeze(1)


def mix_style(sampled, ref_s=None, alpha=0.3, beta=0.7):
    """Sin WAV se usa íntegramente el estilo generado; alpha/beta no intervienen."""
    if not 0 <= alpha <= 1 or not 0 <= beta <= 1:
        raise ValueError("alpha y beta deben estar entre 0 y 1")
    dim = model_params.style_dim
    acoustic, prosody = sampled[:, :dim], sampled[:, dim:]
    if ref_s is not None:
        acoustic = alpha * acoustic + (1 - alpha) * ref_s[:, :dim]
        prosody = beta * prosody + (1 - beta) * ref_s[:, dim:]
    return acoustic, prosody


def inference(text, ref_s=None, alpha = 0.3, beta = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    if not text:
        raise ValueError("El texto no puede estar vacío")
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    if len(tokens) > model.bert.config.max_position_embeddings:
        raise ValueError("Texto demasiado largo; divide el texto en frases")
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 

        s_pred = sample_style(bert_dur, ref_s, diffusion_steps, embedding_scale)


        ref, s = mix_style(s_pred, ref_s, alpha, beta)

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze(0)).clamp(min=1)


        pred_aln_trg = torch.zeros(tokens.shape[-1], int(pred_dur.sum().item()))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
        
    return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later

### Prueba de inferencia: voces y referencia opcional

En multi-speaker se sintetiza por cada entrada de `references`. En single-speaker con `SINGLE_USE_REFERENCE=True`, se prueban esos mismos WAV como estilos de referencia; para empezar usa audios de la persona entrenada. Con `False` se genera una única salida sin abrir referencias.

Para comparar single con/sin referencia, cambia sólo `SINGLE_USE_REFERENCE` en la configuración y vuelve a ejecutar esta prueba. Conserva la semilla, texto y parámetros. `alpha` controla la mezcla acústica y `beta` la prosódica; valores bajos conservan más referencia. Los WAV y textos deben corresponder al idioma/frontend entrenado.


In [62]:
references = {
    "hablante_1": "referencias_inferencia/E2-82.wav",
    "hablante_2": "referencias_inferencia/E1-48.wav",
    "hablante_3": "referencias_inferencia/E2-226.wav",
}
text = "València tindrà noves restriccions horàries per a la pirotècnia recreativa durant les falles de 2027, així com cinc parcs que seran zones lliures de petards. A més, els tendals es muntaran, amb caràcter general, entre l’11 i el 19 de març. Així ho ha informat la regidoria de Festes i Tradicions de la capital del Túria."
#text = "En altres paraules, què té d’especial la neoescorça humana i com es diferencia d’altres espècies?"
#text = " Estic molt content!"
# Misma semilla para comparar estilos con ruido de muestreo reproducible.
SEED = 0
ALPHA, BETA = 0.3, 0.7
DIFFUSION_STEPS = 10
EMBEDDING_SCALE = 1
use_reference = bool(model_params.multispeaker or SINGLE_USE_REFERENCE)
mode = "multi" if model_params.multispeaker else ("single_con_referencia" if use_reference else "single_sin_referencia")
output_dir = Path("Demo/outputs_inference") / mode
output_dir.mkdir(parents=True, exist_ok=True)
cases = references if use_reference else {"voz_entrenada": None}
if not cases:
    raise ValueError("Añade al menos un WAV a references")
print("Modo:", mode)
for speaker, path in cases.items():
    torch.manual_seed(SEED)
    ref_s = compute_style(path) if use_reference else None
    start = time.perf_counter()
    wav = inference(text, ref_s, alpha=ALPHA, beta=BETA,
                    diffusion_steps=DIFFUSION_STEPS, embedding_scale=EMBEDDING_SCALE)
    elapsed = time.perf_counter() - start
    target = output_dir / f"{speaker}.wav"
    sf.write(target, wav, SAMPLE_RATE, subtype="PCM_16")
    print(speaker, target, f"RTF={elapsed / (len(wav) / SAMPLE_RATE):.3f}")
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=True))


Modo: multi
hablante_1 Demo/outputs_inference/multi/hablante_1.wav RTF=0.357


hablante_2 Demo/outputs_inference/multi/hablante_2.wav RTF=0.349


hablante_3 Demo/outputs_inference/multi/hablante_3.wav RTF=0.336


## Demostraciones originales (opcionales)

Se conservan las secciones de hablantes vistos/no vistos, emociones, adaptación zero-shot, narración larga, transferencia de estilo y diversidad. Para reproducirlas literalmente necesitas el modelo LibriTTS, `LANGUAGE="en-us"` y el ZIP de referencias. Para tu modelo valenciano sustituye sus textos y referencias por los tuyos. La calidad zero-shot no está garantizada por activar multispeaker.

En `STinference` se corrige el uso del texto de referencia: el original calculaba su embedding pero no lo utilizaba al muestrear. No garantiza que una etiqueta emocional controle por sí sola la emoción.

#### Basic synthesis (5 diffusion steps, seen speakers)

In [ ]:
text = ''' Noves connexions per a l'aeroport de València. La companyia aèria hongaresa Wizz Air inaugura este dilluns la base a l'aeroport de València. L'aerolínia estrena a partir de hui quatre noves rutes: cinc vols setmanals a Astúries, quatre a Santiago de Compostel·la, dos a Santander i tres a Brasov (Romania), segons va informar ahir diumenge l'aeroport de València a les xarxes socials. '''

In [ ]:
reference_dicts = {}
reference_dicts['696_92939'] = "Demo/reference_audio/696_92939_000016_000006.wav"
reference_dicts['1789_142896'] = "Demo/reference_audio/1789_142896_000022_000005.wav"

In [ ]:
start = time.time()
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / SAMPLE_RATE)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized:')
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=SAMPLE_RATE, normalize=False))

#### With higher diffusion steps (more diverse)

Since the sampler is ancestral, the higher the stpes, the more diverse the samples are, with the cost of slower synthesis speed.

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=10, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / SAMPLE_RATE)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized:')
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=SAMPLE_RATE, normalize=False))

#### Basic synthesis (5 diffusion steps, umseen speakers)
The following samples are to reproduce samples in [Section 4](https://styletts2.github.io/#libri) of the demo page. All spsakers are unseen during training. You can compare the generated samples to popular zero-shot TTS models like Vall-E and NaturalSpeech 2.

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['1221-135767'] = ("Demo/reference_audio/1221-135767-0014.wav", "Yea, his honourable worship is within, but he hath a godly minister or two with him, and likewise a leech.")
reference_dicts['5639-40744'] = ("Demo/reference_audio/5639-40744-0020.wav", "Thus did this humane and right minded father comfort his unhappy daughter, and her mother embracing her again, did all she could to soothe her feelings.")
reference_dicts['908-157963'] = ("Demo/reference_audio/908-157963-0027.wav", "And lay me down in my cold bed and leave my shining lot.")
reference_dicts['4077-13754'] = ("Demo/reference_audio/4077-13754-0000.wav", "The army found the people in poverty and left them in comparative wealth.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / SAMPLE_RATE)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized: ' + text)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=SAMPLE_RATE, normalize=False))

### Speech expressiveness

The following section recreates the samples shown in [Section 6](https://styletts2.github.io/#emo) of the demo page. The speaker reference used is `1221-135767-0014.wav`, which is unseen during training. 

#### With `embedding_scale=1`
This is the classifier-free guidance scale. The higher the scale, the more conditional the style is to the input text and hence more emotional.



In [ ]:
ref_s = compute_style("Demo/reference_audio/1221-135767-0014.wav")

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=1)
    print(k + ": ")
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

#### With `embedding_scale=2`

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    noise = torch.randn(1,1,256).to(device)
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=2)
    print(k + ": ")
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

#### With `embedding_scale=2, alpha = 0.5, beta = 0.9`
`alpha` and `beta` is the factor to determine much we use the style sampled based on the text instead of the reference. The higher the value of `alpha` and `beta`, the more suitable the style it is to the text but less similar to the reference. Using higher beta makes the synthesized speech more emotional, at the cost of lower similarity to the reference. `alpha` determines the timbre of the speaker while `beta` determines the prosody. 

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    noise = torch.randn(1,1,256).to(device)
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.5, beta=0.9, embedding_scale=2)
    print(k + ": ")
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

### Zero-shot speaker adaptation
This section recreates the "Acoustic Environment Maintenance" and "Speaker’s Emotion Maintenance" demo in [Section 4](https://styletts2.github.io/#libri) of the demo page. You can compare the generated samples to popular zero-shot TTS models like Vall-E. Note that the model was trained only on LibriTTS, which is about 250 times fewer data compared to those used to trian Vall-E with similar or better effect for these maintainance. 

#### Acoustic Environment Maintenance

Since we want to maintain the acoustic environment in the speaker (timbre), we set  `alpha = 0` to make the speaker as closer to the reference as possible while only changing the prosody according to the text.  

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['3'] = ("Demo/reference_audio/3.wav", "As friends thing I definitely I've got more male friends.")
reference_dicts['4'] = ("Demo/reference_audio/4.wav", "Everything is run by computer but you got to know how to think before you can do a computer.")
reference_dicts['5'] = ("Demo/reference_audio/5.wav", "Then out in LA you guys got a whole another ball game within California to worry about.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.0, beta=0.5, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / SAMPLE_RATE)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print('Synthesized: ' + text)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=SAMPLE_RATE, normalize=False))

#### Speaker’s Emotion Maintenance

Since we want to maintain the emotion in the speaker (prosody), we set  `beta = 0.1` to make the speaker as closer to the reference as possible while having some diversity thruogh the slight timbre change.

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['Anger'] = ("Demo/reference_audio/anger.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Sleepy'] = ("Demo/reference_audio/sleepy.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Amused'] = ("Demo/reference_audio/amused.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Disgusted'] = ("Demo/reference_audio/disgusted.wav", "We have to reduce the number of plastic bags.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.1, diffusion_steps=10, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / SAMPLE_RATE)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized: ' + text)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=SAMPLE_RATE, normalize=False))

### Longform Narration

This section includes basic implementation of Algorithm 1 in the paper for consistent longform audio generation. The example passage is taken from [Section 5](https://styletts2.github.io/#long) of the demo page.

In [ ]:
passage = '''If the supply of fruit is greater than the family needs, it may be made a source of income by sending the fresh fruit to the market if there is one near enough, or by preserving, canning, and making jelly for sale. To make such an enterprise a success the fruit and work must be first class. There is magic in the word "Homemade," when the product appeals to the eye and the palate; but many careless and incompetent people have found to their sorrow that this word has not magic enough to float inferior goods on the market. As a rule large canning and preserving establishments are clean and have the best appliances, and they employ chemists and skilled labor. The home product must be very good to compete with the attractive goods that are sent out from such establishments. Yet for first class home made products there is a market in all large cities. All first-class grocers have customers who purchase such goods.'''

In [ ]:
def LFinference(text, s_prev, ref_s=None, alpha = 0.3, beta = 0.7, t = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    if not text:
        raise ValueError("El texto no puede estar vacío")
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    ps = ps.replace('``', '"')
    ps = ps.replace("''", '"')

    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    if len(tokens) > model.bert.config.max_position_embeddings:
        raise ValueError("Texto demasiado largo; divide el texto en frases")
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 

        s_pred = sample_style(bert_dur, ref_s, diffusion_steps, embedding_scale)
        
        if s_prev is not None:
            # convex combination of previous and current style
            s_pred = t * s_prev + (1 - t) * s_pred
        
        ref, s = mix_style(s_pred, ref_s, alpha, beta)

        s_pred = torch.cat([ref, s], dim=-1)

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze(0)).clamp(min=1)


        pred_aln_trg = torch.zeros(tokens.shape[-1], int(pred_dur.sum().item()))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
        
    return out.squeeze().cpu().numpy()[..., :-100], s_pred # weird pulse at the end of the model, need to be fixed later

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
s_ref = compute_style(path)
sentences = passage.split('.') # simple split by comma
wavs = []
s_prev = None
for text in sentences:
    if text.strip() == "": continue
    text += '.' # add it back
    
    wav, s_prev = LFinference(text, 
                              s_prev, 
                              s_ref, 
                              alpha = 0.3, 
                              beta = 0.9,  # make it more suitable for the text
                              t = 0.7, 
                              diffusion_steps=10, embedding_scale=1.5)
    wavs.append(wav)
print('Synthesized: ')
display(ipd.Audio(np.concatenate(wavs), rate=SAMPLE_RATE, normalize=False))
print('Reference: ')
display(ipd.Audio(path, rate=SAMPLE_RATE, normalize=False))

### Style Transfer

The following section demostrates the style transfer capacity for unseen speakers in [Section 6](https://styletts2.github.io/#emo) of the demo page. For this, we set `alpha=0.5, beta = 0.9` for the most pronounced effects (mostly using the sampled style). 

In [ ]:
def STinference(text, ref_s, ref_text, alpha = 0.3, beta = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    if not text:
        raise ValueError("El texto no puede estar vacío")
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)

    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    if len(tokens) > model.bert.config.max_position_embeddings:
        raise ValueError("Texto demasiado largo; divide el texto en frases")
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    ref_text = ref_text.strip()
    ps = global_phonemizer.phonemize([ref_text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)

    ref_tokens = textclenaer(ps)
    ref_tokens.insert(0, 0)
    if len(ref_tokens) > model.bert.config.max_position_embeddings:
        raise ValueError("Referencia textual demasiado larga")
    ref_tokens = torch.LongTensor(ref_tokens).to(device).unsqueeze(0)
    
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 
        
        ref_input_lengths = torch.LongTensor([ref_tokens.shape[-1]]).to(device)
        ref_text_mask = length_to_mask(ref_input_lengths).to(device)
        ref_bert_dur = model.bert(ref_tokens, attention_mask=(~ref_text_mask).int())
        s_pred = sample_style(ref_bert_dur, ref_s, diffusion_steps, embedding_scale)


        ref, s = mix_style(s_pred, ref_s, alpha, beta)

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze(0)).clamp(min=1)


        pred_aln_trg = torch.zeros(tokens.shape[-1], int(pred_dur.sum().item()))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
        
    return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later

In [ ]:
# reference texts to sample styles

ref_texts = {}
ref_texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
ref_texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
ref_texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
ref_texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

In [ ]:
path = "Demo/reference_audio/1221-135767-0014.wav"
s_ref = compute_style(path)

text = "Yea, his honourable worship is within, but he hath a godly minister or two with him, and likewise a leech."
for k,v in ref_texts.items():
    wav = STinference(text, s_ref, v, diffusion_steps=10, alpha=0.5, beta=0.9, embedding_scale=1.5)
    print(k + ": ")
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

### Speech diversity

This section reproduces samples in [Section 7](https://styletts2.github.io/#var) of the demo page. 

`alpha` and `beta` determine the diversity of the synthesized speech. There are two extreme cases:
- If `alpha = 1` and `beta = 1`, the synthesized speech sounds the most dissimilar to the reference speaker, but it is also the most diverse (each time you synthesize a speech it will be totally different). 
- If `alpha = 0` and `beta = 0`, the synthesized speech sounds the most siimlar to the reference speaker, but it is deterministic (i.e., the sampled style is not used for speech synthesis). 


#### Default setting (`alpha = 0.3, beta=0.7`)
This setting uses 70% of the reference timbre and 30% of the reference prosody and use the diffusion model to sample them based on the text. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=1)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

#### Less diverse setting (`alpha = 0.1, beta=0.3`)
This setting uses 90% of the reference timbre and 70% of the reference prosody. This makes it more similar to the reference speaker at cost of less diverse samples. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.1, beta=0.3, embedding_scale=1)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

#### More diverse setting (`alpha = 0.5, beta=0.95`)
This setting uses 50% of the reference timbre and 5% of the reference prosody (so it uses 95% of the sampled prosody, which makes it more diverse), but this makes it more dissimilar to the reference speaker.  

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.5, beta=0.95, embedding_scale=1)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

#### Extreme setting (`alpha = 1, beta=1`)
This setting uses 0% of the reference timbre and prosody and use the diffusion model to sample the entire style. This makes the speaker very dissimilar to the reference speaker. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=1, beta=1, embedding_scale=1)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

#### Sin variación (`alpha=0, beta=0`)

Se utiliza el 100% del timbre y la prosodia de referencia; se descarta el estilo muestreado.

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0, beta=0, embedding_scale=1)
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))

### Extra fun!

Here we clone some of the authors' voice of the StyleTTS 2 papers with a few seconds of the recording in the wild. None of the voices is in the dataset and all authors agreed to have their voices cloned here.

In [ ]:
text = ''' StyleTTS 2 is a text to speech model that leverages style diffusion and adversarial training with large speech language models to achieve human level text to speech synthesis. '''

In [ ]:
reference_dicts = {}
reference_dicts['Yinghao'] = "Demo/reference_audio/Yinghao.wav"
reference_dicts['Gavin'] = "Demo/reference_audio/Gavin.wav"
reference_dicts['Vinay'] = "Demo/reference_audio/Vinay.wav"
reference_dicts['Nima'] = "Demo/reference_audio/Nima.wav"

In [ ]:
start = time.time()
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    
    wav = inference(text, ref_s, alpha=0.1, beta=0.5, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / SAMPLE_RATE)
    print('Speaker: ' + k)
    import IPython.display as ipd
    print('Synthesized:')
    display(ipd.Audio(wav, rate=SAMPLE_RATE, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=SAMPLE_RATE, normalize=False))